# Temporal VUS model training and validation
This notebook models future reclassification of starting VUS variants, not five-class clinical diagnosis. Overall accuracy is dominated by unchanged VUS records: report balanced accuracy, macro F1 and per-target ranking alongside it. Selection uses development validation only; no test predictions are made here. Recurring variants across chronological splits limit generalization claims. Previously evaluated test intervals are not fresh external tests.


In [1]:
from pathlib import Path
import json
import pandas as pd
import sklearn

PROJECT_ROOT = Path(r"D:\GeneVISTA")
processed_folder = PROJECT_ROOT / "data" / "processed"
feature_folder = processed_folder / "model_features"
label_folder = processed_folder / "model_labels"

model_folder = PROJECT_ROOT / "models" / "baseline_v1"
model_folder.mkdir(parents=True, exist_ok=True)

feature_manifest = pd.read_csv(
    feature_folder / "feature_manifest.csv",
    dtype="string", keep_default_na=False
)

feature_config = json.loads(
    (feature_folder / "baseline_v1_config.json").read_text(
        encoding="utf-8"
    )
)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_validation import load_label_manifest
from src.training_validation import (
    load_split, validate_config, validate_manifest, fit_converged,
    ranking_metrics, safe_ratio, select_primary,
)
label_manifest = load_label_manifest(label_folder, processed_folder)
validate_config(feature_config)
validate_manifest(feature_manifest, label_manifest)

categorical_features = feature_config["categorical_features"]
numeric_features = feature_config["numeric_features"]

# Training and validation only; reserve test data for final evaluation.
development_manifest = feature_manifest.loc[
    feature_manifest["split"].isin(["train", "validation"])
].copy()

for row in development_manifest.itertuples(index=False):
    for file in [
        feature_folder / row.feature_file,
        label_folder / row.label_file
    ]:
        if not file.exists():
            raise FileNotFoundError(file)

print("MODEL TRAINING SETUP READY")
print(f"scikit-learn version: {sklearn.__version__}")
display(development_manifest)

MODEL TRAINING SETUP READY
scikit-learn version: 1.9.0
                  feature_file                            label_file feature_snapshot       split     rows
0  vus_features_2022-01.csv.gz  vus_labels_2022-01_to_2023-01.csv.gz          2022-01       train   407315
1  vus_features_2023-01.csv.gz  vus_labels_2023-01_to_2024-01.csv.gz          2023-01       train   615192
2  vus_features_2024-01.csv.gz  vus_labels_2024-01_to_2025-01.csv.gz          2024-01  validation  1121932


In [3]:
import numpy as np
model_features = categorical_features + numeric_features
join_keys = ["VariationID", "feature_snapshot"]

def load_model_split(split_name):
    X, y, keys = load_split(processed_folder, feature_manifest, label_manifest,
                            feature_config, split_name)
    print(f"{split_name}: {len(X):,} examples")
    return X, y, keys

X_train, y_train, train_keys = load_model_split("train")
X_validation, y_validation, validation_keys = load_model_split("validation")
assert list(X_train.columns) == list(X_validation.columns)
assert not set(feature_config["excluded_from_model_inputs"]) & set(X_train.columns)
overlap_count = int(validation_keys["VariationID"].isin(set(train_keys["VariationID"])).sum())
print(f"Validation variants also seen in training: {overlap_count:,} / {len(validation_keys):,}")
print("Chronological evaluation with recurring variants; not unseen-variant validation.")
print("TRAINING CLASS COUNTS")
print(y_train.value_counts().to_string())


train: 1,022,507 examples
validation: 1,121,932 examples
Validation variants also seen in training: 589,120 / 1,121,932
Chronological evaluation with recurring variants; not unseen-variant validation.
TRAINING CLASS COUNTS
target
stayed_vus           1008079
became_benign          11244
became_pathogenic       3184


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

class_names = feature_config["target_classes"]

dummy_model = DummyClassifier(strategy="most_frequent")

# The dummy model does not need actual feature values.
dummy_model.fit(
    np.zeros((len(y_train), 1), dtype=np.uint8),
    y_train
)

dummy_predictions = dummy_model.predict(
    np.zeros((len(y_validation), 1), dtype=np.uint8)
)

dummy_results = {
    "model": "majority_class",
    "accuracy": accuracy_score(y_validation, dummy_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, dummy_predictions
    ),
    "macro_f1": f1_score(
        y_validation,
        dummy_predictions,
        labels=class_names,
        average="macro",
        zero_division=0
    )
}

print("MAJORITY-CLASS VALIDATION BASELINE")
display(pd.DataFrame([dummy_results]))

print(
    classification_report(
        y_validation,
        dummy_predictions,
        labels=class_names,
        digits=4,
        zero_division=0
    )
)

display(
    pd.DataFrame(
        confusion_matrix(
            y_validation,
            dummy_predictions,
            labels=class_names
        ),
        index=[f"actual_{name}" for name in class_names],
        columns=[f"predicted_{name}" for name in class_names]
    )
)

MAJORITY-CLASS VALIDATION BASELINE
            model  accuracy  balanced_accuracy  macro_f1
0  majority_class  0.992786           0.333333  0.332127
                   precision    recall  f1-score   support

       stayed_vus     0.9928    1.0000    0.9964   1113838
    became_benign     0.0000    0.0000    0.0000      6325
became_pathogenic     0.0000    0.0000    0.0000      1769

         accuracy                         0.9928   1121932
        macro avg     0.3309    0.3333    0.3321   1121932
     weighted avg     0.9856    0.9928    0.9892   1121932

                          predicted_stayed_vus  predicted_became_benign  predicted_became_pathogenic
actual_stayed_vus                      1113838                        0                            0
actual_became_benign                      6325                        0                            0
actual_became_pathogenic                  1769                        0                            0


In [5]:
from time import perf_counter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        keep_empty_features=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="missing",
        keep_empty_features=True
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        class_weight="balanced",
        C=1.0,
        max_iter=500
    ))
])

print("Training logistic regression...")
started = perf_counter()

# Imputation, scaling, and encoding are fitted on training data only.
fit_converged(logistic_model, X_train, y_train)

elapsed = perf_counter() - started
classifier = logistic_model.named_steps["classifier"]

print(f"Training finished in {elapsed / 60:.1f} minutes.")
print(f"Solver iterations: {classifier.n_iter_.max()}")

if classifier.n_iter_.max() >= classifier.max_iter:
    print("Iteration limit reached. Share any convergence warning.")

Training logistic regression...
Training finished in 0.2 minutes.
Solver iterations: 115


In [6]:
print("Predicting validation outcomes...")

logistic_predictions = logistic_model.predict(X_validation)

logistic_results = {
    "model": "balanced_logistic_regression",
    "accuracy": accuracy_score(
        y_validation, logistic_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, logistic_predictions
    ),
    "macro_f1": f1_score(
        y_validation,
        logistic_predictions,
        labels=class_names,
        average="macro",
        zero_division=0
    )
}

print("\nVALIDATION COMPARISON")
display(pd.DataFrame([dummy_results, logistic_results]))

print(
    classification_report(
        y_validation,
        logistic_predictions,
        labels=class_names,
        digits=4,
        zero_division=0
    )
)

display(
    pd.DataFrame(
        confusion_matrix(
            y_validation,
            logistic_predictions,
            labels=class_names
        ),
        index=[f"actual_{name}" for name in class_names],
        columns=[f"predicted_{name}" for name in class_names]
    )
)

Predicting validation outcomes...

VALIDATION COMPARISON
                          model  accuracy  balanced_accuracy  macro_f1
0                majority_class  0.992786           0.333333  0.332127
1  balanced_logistic_regression  0.338539           0.470932  0.179214
                   precision    recall  f1-score   support

       stayed_vus     0.9958    0.3363    0.5028   1113838
    became_benign     0.0067    0.7328    0.0133      6325
became_pathogenic     0.0111    0.3437    0.0216      1769

         accuracy                         0.3385   1121932
        macro avg     0.3379    0.4709    0.1792   1121932
     weighted avg     0.9887    0.3385    0.4993   1121932

                          predicted_stayed_vus  predicted_became_benign  predicted_became_pathogenic
actual_stayed_vus                       374575                   685700                        53563
actual_became_benign                      1227                     4635                          463
actual_beca

In [7]:
from sklearn.base import clone

# Same preprocessing and model settings; change only class weighting.
unweighted_model = clone(logistic_model)
unweighted_model.set_params(classifier__class_weight=None)

print("Training unweighted logistic regression...")
started = perf_counter()

fit_converged(unweighted_model, X_train, y_train)

print(f"Training finished in {(perf_counter() - started) / 60:.1f} minutes.")

iterations = unweighted_model.named_steps["classifier"].n_iter_.max()
print(f"Solver iterations: {iterations}")

if iterations >= unweighted_model.named_steps["classifier"].max_iter:
    print("Iteration limit reached. Share any convergence warning.")

unweighted_predictions = unweighted_model.predict(X_validation)

unweighted_results = {
    "model": "unweighted_logistic_regression",
    "accuracy": accuracy_score(
        y_validation, unweighted_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, unweighted_predictions
    ),
    "macro_f1": f1_score(
        y_validation,
        unweighted_predictions,
        labels=class_names,
        average="macro",
        zero_division=0
    )
}

display(pd.DataFrame([
    dummy_results,
    logistic_results,
    unweighted_results
]))

print(
    classification_report(
        y_validation,
        unweighted_predictions,
        labels=class_names,
        digits=4,
        zero_division=0
    )
)

Training unweighted logistic regression...
Training finished in 0.1 minutes.
Solver iterations: 30
                            model  accuracy  balanced_accuracy  macro_f1
0                  majority_class  0.992786           0.333333  0.332127
1    balanced_logistic_regression  0.338539           0.470932  0.179214
2  unweighted_logistic_regression  0.992778           0.333331  0.332125
                   precision    recall  f1-score   support

       stayed_vus     0.9928    1.0000    0.9964   1113838
    became_benign     0.0000    0.0000    0.0000      6325
became_pathogenic     0.0000    0.0000    0.0000      1769

         accuracy                         0.9928   1121932
        macro avg     0.3309    0.3333    0.3321   1121932
     weighted avg     0.9856    0.9928    0.9892   1121932



In [8]:
from sklearn.metrics import average_precision_score

ranking_results = []

for model_name, model in [
    ("balanced_logistic", logistic_model),
    ("unweighted_logistic", unweighted_model)
]:
    probabilities = model.predict_proba(X_validation)
    model_classes = list(model.named_steps["classifier"].classes_)

    for target in ["became_benign", "became_pathogenic"]:
        actual = y_validation.eq(target).to_numpy()
        scores = probabilities[:, model_classes.index(target)]

        prevalence, average_precision, lift = ranking_metrics(actual, scores)

        ranking_results.append({
            "model": model_name,
            "target": target,
            "validation_prevalence": prevalence,
            "average_precision": average_precision,
            "AP_divided_by_prevalence": lift
        })

ranking_summary = pd.DataFrame(ranking_results)

print("VALIDATION RANKING QUALITY")
display(ranking_summary)

VALIDATION RANKING QUALITY
                 model             target  validation_prevalence  average_precision  AP_divided_by_prevalence
0    balanced_logistic      became_benign               0.005638           0.005165                  0.916113
1    balanced_logistic  became_pathogenic               0.001577           0.008676                  5.502771
2  unweighted_logistic      became_benign               0.005638           0.005218                  0.925557
3  unweighted_logistic  became_pathogenic               0.001577           0.007188                  4.558872


In [9]:
from datetime import datetime
import joblib

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
experiment_folder = model_folder / f"snapshot_baseline_{run_name}"
experiment_folder.mkdir(parents=True, exist_ok=False)

comparison = pd.DataFrame([
    dummy_results,
    logistic_results,
    unweighted_results
])

comparison.to_csv(
    experiment_folder / "validation_metrics.csv",
    index=False
)

ranking_summary.to_csv(
    experiment_folder / "validation_ranking.csv",
    index=False
)

joblib.dump(
    logistic_model,
    experiment_folder / "balanced_logistic.joblib"
)

joblib.dump(
    unweighted_model,
    experiment_folder / "unweighted_logistic.joblib"
)

experiment_notes = {
    "feature_config": feature_config,
    "sklearn_version": sklearn.__version__,
    "training_rows": len(y_train),
    "validation_rows": len(y_validation),
    "test_evaluated": False,
    "status": "Experimental baseline; not selected for deployment",
    "next_experiment": "Add features from prior snapshots only"
}

(experiment_folder / "experiment_notes.json").write_text(
    json.dumps(experiment_notes, indent=2),
    encoding="utf-8"
)

# Verify both complete pipelines and class probabilities after serialization.
for filename, original in [("balanced_logistic.joblib", logistic_model),
                           ("unweighted_logistic.joblib", unweighted_model)]:
    restored = joblib.load(experiment_folder / filename)
    sample = X_validation.iloc[:1000]
    np.testing.assert_array_equal(restored.classes_, original.classes_)
    np.testing.assert_allclose(restored.predict_proba(sample), original.predict_proba(sample), rtol=1e-12, atol=1e-12)

print("BASELINE EXPERIMENT SAVED AND RELOAD CHECK PASSED")
print(experiment_folder)

BASELINE EXPERIMENT SAVED AND RELOAD CHECK PASSED
D:\GeneVISTA\models\baseline_v1\snapshot_baseline_20260923_095802_115877


In [10]:
temporal_folder = processed_folder / "temporal_features_v1"

temporal_manifest = pd.read_csv(
    temporal_folder / "temporal_manifest.csv",
    dtype="string", keep_default_na=False
)

temporal_config = json.loads(
    (temporal_folder / "temporal_config.json").read_text(
        encoding="utf-8"
    )
)

temporal_categorical = temporal_config["categorical_features"]
temporal_numeric = temporal_config["numeric_features"]
temporal_inputs = temporal_categorical + temporal_numeric


validate_config(temporal_config, temporal=True)
validate_manifest(temporal_manifest, label_manifest, feature_manifest)

def load_temporal_split(split_name):
    X, y, keys = load_split(processed_folder, temporal_manifest, label_manifest,
                            temporal_config, split_name, temporal=True)
    print(f"{split_name}: {len(X):,} rows, {X.shape[1]} features")
    return X, y, keys

X_train_temporal, y_train_temporal, temporal_train_keys = load_temporal_split("train")
X_validation_temporal, y_validation_temporal, temporal_validation_keys = load_temporal_split("validation")
pd.testing.assert_frame_equal(train_keys, temporal_train_keys)
pd.testing.assert_frame_equal(validation_keys, temporal_validation_keys)
pd.testing.assert_series_equal(y_train, y_train_temporal)
pd.testing.assert_series_equal(y_validation, y_validation_temporal)
print("TEMPORAL MODEL DATA READY")
print(y_train_temporal.value_counts().to_string())


train: 1,022,507 rows, 12 features
validation: 1,121,932 rows, 12 features
TEMPORAL MODEL DATA READY
target
stayed_vus           1008079
became_benign          11244
became_pathogenic       3184


In [11]:
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

temporal_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="median",
                    keep_empty_features=True
                )),
                ("scaler", StandardScaler())
            ]),
            temporal_numeric
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="constant",
                    fill_value="missing",
                    keep_empty_features=True
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ))
            ]),
            temporal_categorical
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

temporal_models = {}

for name, weighting in [
    ("temporal_balanced", "balanced"),
    ("temporal_unweighted", None)
]:
    model = Pipeline([
        ("preprocessor", clone(temporal_preprocessor)),
        ("classifier", LogisticRegression(
            solver="lbfgs",
            class_weight=weighting,
            C=1.0,
            max_iter=500
        ))
    ])

    print(f"Training {name}...")
    started = perf_counter()

    fit_converged(model, X_train_temporal, y_train_temporal)
    temporal_models[name] = model

    classifier = model.named_steps["classifier"]
    iterations = int(classifier.n_iter_.max())

    print(f"  Time: {(perf_counter() - started) / 60:.1f} minutes")
    print(f"  Iterations: {iterations}")

    if iterations >= classifier.max_iter:
        print("  Iteration limit reached; share any convergence warning.")

Training temporal_balanced...
  Time: 0.4 minutes
  Iterations: 187
Training temporal_unweighted...
  Time: 0.1 minutes
  Iterations: 33


In [12]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    average_precision_score
)

temporal_results = []
temporal_ranking_results = []

target_classes = temporal_config["target_classes"]

for name, model in temporal_models.items():
    print(f"\nEvaluating {name}...")

    probabilities = model.predict_proba(X_validation_temporal)
    model_classes = model.named_steps["classifier"].classes_
    predictions = model_classes[probabilities.argmax(axis=1)]

    temporal_results.append({
        "model": name,
        "accuracy": accuracy_score(
            y_validation_temporal, predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_validation_temporal, predictions
        ),
        "macro_f1": f1_score(
            y_validation_temporal,
            predictions,
            labels=target_classes,
            average="macro",
            zero_division=0
        )
    })

    print(
        classification_report(
            y_validation_temporal,
            predictions,
            labels=target_classes,
            digits=4,
            zero_division=0
        )
    )

    for target in ["became_benign", "became_pathogenic"]:
        actual = y_validation_temporal.eq(target).to_numpy()
        column = list(model_classes).index(target)
        scores = probabilities[:, column]

        prevalence, ap, lift = ranking_metrics(actual, scores)

        temporal_ranking_results.append({
            "model": name,
            "target": target,
            "validation_prevalence": prevalence,
            "average_precision": ap,
            "AP_divided_by_prevalence": lift
        })

temporal_metrics = pd.DataFrame(temporal_results)
temporal_ranking = pd.DataFrame(temporal_ranking_results)

print("\nBASELINE VS TEMPORAL — VALIDATION METRICS")
display(
    pd.concat([
        pd.DataFrame([
            dummy_results,
            logistic_results,
            unweighted_results
        ]),
        temporal_metrics
    ], ignore_index=True)
)

print("\nBASELINE VS TEMPORAL — RANKING QUALITY")
display(
    pd.concat([
        ranking_summary,
        temporal_ranking
    ], ignore_index=True)
)


Evaluating temporal_balanced...
                   precision    recall  f1-score   support

       stayed_vus     0.9987    0.2481    0.3974   1113838
    became_benign     0.0072    0.8975    0.0142      6325
became_pathogenic     0.0108    0.3262    0.0208      1769

         accuracy                         0.2518   1121932
        macro avg     0.3389    0.4906    0.1441   1121932
     weighted avg     0.9915    0.2518    0.3946   1121932


Evaluating temporal_unweighted...
                   precision    recall  f1-score   support

       stayed_vus     0.9928    1.0000    0.9964   1113838
    became_benign     0.0000    0.0000    0.0000      6325
became_pathogenic     0.0000    0.0000    0.0000      1769

         accuracy                         0.9928   1121932
        macro avg     0.3309    0.3333    0.3321   1121932
     weighted avg     0.9856    0.9928    0.9892   1121932


BASELINE VS TEMPORAL — VALIDATION METRICS
                            model  accuracy  balanced_acc

In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from time import perf_counter

# Encode categories using training data only.
# Unknown validation categories are represented as missing.
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", temporal_numeric),
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=np.nan,
                encoded_missing_value=np.nan,
                dtype=np.float64
            ),
            temporal_categorical
        )
    ],
    remainder="drop",
    sparse_threshold=0
)

# Tell the model which transformed columns are categorical.
category_mask = (
    [False] * len(temporal_numeric)
    + [True] * len(temporal_categorical)
)

for column in temporal_categorical:
    assert X_train_temporal[column].nunique(dropna=True) <= 255

boosted_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=150,
        max_leaf_nodes=15,
        min_samples_leaf=100,
        l2_regularization=1.0,
        categorical_features=category_mask,
        class_weight=None,
        early_stopping=False,
        random_state=42
    ))
])

# Fixed iteration count avoids an automatic random validation split.
print("Training temporal gradient boosting...")
started = perf_counter()

boosted_model.fit(X_train_temporal, y_train_temporal)

print(f"Training finished in {(perf_counter() - started) / 60:.1f} minutes.")

Training temporal gradient boosting...
Training finished in 0.2 minutes.


In [14]:
boosted_probabilities = boosted_model.predict_proba(
    X_validation_temporal
)

boosted_classes = boosted_model.named_steps["classifier"].classes_
boosted_predictions = boosted_classes[
    boosted_probabilities.argmax(axis=1)
]

boosted_results = {
    "model": "temporal_gradient_boosting",
    "accuracy": accuracy_score(
        y_validation_temporal, boosted_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation_temporal, boosted_predictions
    ),
    "macro_f1": f1_score(
        y_validation_temporal,
        boosted_predictions,
        labels=target_classes,
        average="macro",
        zero_division=0
    )
}

print("VALIDATION METRICS")
display(pd.concat([
    pd.DataFrame([dummy_results]),
    temporal_metrics,
    pd.DataFrame([boosted_results])
], ignore_index=True))

print(
    classification_report(
        y_validation_temporal,
        boosted_predictions,
        labels=target_classes,
        digits=4,
        zero_division=0
    )
)

boosted_ranking_rows = []

for target in ["became_benign", "became_pathogenic"]:
    actual = y_validation_temporal.eq(target).to_numpy()
    column = list(boosted_classes).index(target)
    scores = boosted_probabilities[:, column]

    prevalence, ap, lift = ranking_metrics(actual, scores)

    boosted_ranking_rows.append({
        "model": "temporal_gradient_boosting",
        "target": target,
        "validation_prevalence": prevalence,
        "average_precision": ap,
        "AP_divided_by_prevalence": lift
    })

boosted_ranking = pd.DataFrame(boosted_ranking_rows)

print("VALIDATION RANKING COMPARISON")
display(pd.concat([
    temporal_ranking,
    boosted_ranking
], ignore_index=True))

VALIDATION METRICS
                        model  accuracy  balanced_accuracy  macro_f1
0              majority_class  0.992786           0.333333  0.332127
1           temporal_balanced  0.251837           0.490592  0.144149
2         temporal_unweighted  0.992784           0.333333  0.332126
3  temporal_gradient_boosting  0.992786           0.333333  0.332127
                   precision    recall  f1-score   support

       stayed_vus     0.9928    1.0000    0.9964   1113838
    became_benign     0.0000    0.0000    0.0000      6325
became_pathogenic     0.0000    0.0000    0.0000      1769

         accuracy                         0.9928   1121932
        macro avg     0.3309    0.3333    0.3321   1121932
     weighted avg     0.9856    0.9928    0.9892   1121932

VALIDATION RANKING COMPARISON
                        model             target  validation_prevalence  average_precision  AP_divided_by_prevalence
0           temporal_balanced      became_benign               0.005638  

In [15]:
# Fixed review-list sizes for comparing models.
review_sizes = [100, 1000, 5000]
review_results = []

models_to_compare = {
    "temporal_balanced": temporal_models["temporal_balanced"],
    "temporal_unweighted": temporal_models["temporal_unweighted"],
    "temporal_gradient_boosting": boosted_model
}

# Random, reproducible ordering prevents row order from deciding score ties.
tie_order = np.random.default_rng(42).permutation(
    len(y_validation_temporal)
)

for model_name, model in models_to_compare.items():
    print(f"Checking review lists: {model_name}...")

    probabilities = model.predict_proba(X_validation_temporal)
    classes = list(model.named_steps["classifier"].classes_)

    for target in ["became_benign", "became_pathogenic"]:
        actual = y_validation_temporal.eq(target).to_numpy()
        scores = probabilities[:, classes.index(target)]

        total_positives = int(actual.sum())
        prevalence = float(actual.mean())

        ranked_positions = tie_order[
            np.argsort(-scores[tie_order], kind="stable")
        ]

        for size in sorted({min(requested_size, len(actual)) for requested_size in review_sizes}):
            selected = ranked_positions[:size]

            true_positives = int(actual[selected].sum())
            precision = safe_ratio(true_positives, size)
            recall = safe_ratio(true_positives, total_positives)

            review_results.append({
                "model": model_name,
                "target": target,
                "review_size": size,
                "true_reclassifications": true_positives,
                "other_outcomes": size - true_positives,
                "precision_percent": round(100 * precision, 2),
                "recall_percent": round(100 * recall, 2),
                "lift_over_prevalence": round(
                    safe_ratio(precision, prevalence), 2
                )
            })

review_summary = pd.DataFrame(review_results)

for target in ["became_benign", "became_pathogenic"]:
    print(f"\nREVIEW-LIST RESULTS: {target}")
    print(
        review_summary.loc[
            review_summary["target"].eq(target)
        ].drop(columns="target").to_string(index=False)
    )

Checking review lists: temporal_balanced...
Checking review lists: temporal_unweighted...
Checking review lists: temporal_gradient_boosting...

REVIEW-LIST RESULTS: became_benign
                     model  review_size  true_reclassifications  other_outcomes  precision_percent  recall_percent  lift_over_prevalence
         temporal_balanced          100                       3              97               3.00            0.05                  5.32
         temporal_balanced         1000                      16             984               1.60            0.25                  2.84
         temporal_balanced         5000                      77            4923               1.54            1.22                  2.73
       temporal_unweighted          100                       2              98               2.00            0.03                  3.55
       temporal_unweighted         1000                       7             993               0.70            0.11                  1.24

In [16]:
from datetime import datetime
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

evaluation_bundle = (
    PROJECT_ROOT / "models" / "temporal_v1" / run_name
)
evaluation_bundle.mkdir(parents=True, exist_ok=False)

all_ranking = pd.concat([temporal_ranking, boosted_ranking], ignore_index=True)
primary_name, selection_table = select_primary(all_ranking, models_to_compare)
comparator_name = ("temporal_balanced" if primary_name != "temporal_balanced"
                   else "temporal_gradient_boosting")
primary_key = "primary_" + primary_name
comparator_key = "comparator_" + comparator_name
frozen_models = {
    primary_key: models_to_compare[primary_name],
    comparator_key: models_to_compare[comparator_name],
}
selection_table.to_csv(evaluation_bundle / "model_selection.csv", index=False)
display(selection_table)

# Save complete pipelines, including their fitted preprocessing.
for name, model in frozen_models.items():
    model_file = evaluation_bundle / f"{name}.joblib"
    joblib.dump(model, model_file)

    reloaded = joblib.load(model_file)
    sample = X_validation_temporal.iloc[:1000]

    np.testing.assert_allclose(
        reloaded.predict_proba(sample),
        model.predict_proba(sample),
        rtol=1e-12,
        atol=1e-12
    )

validation_metrics = pd.concat([
    temporal_metrics,
    pd.DataFrame([boosted_results])
], ignore_index=True)

validation_ranking = pd.concat([
    temporal_ranking,
    boosted_ranking
], ignore_index=True)

validation_metrics.to_csv(
    evaluation_bundle / "validation_metrics.csv",
    index=False
)

validation_ranking.to_csv(
    evaluation_bundle / "validation_ranking.csv",
    index=False
)

review_summary.to_csv(
    evaluation_bundle / "validation_review_lists.csv",
    index=False
)

# Record exact feature and label files for the held-out interval.
test_rows = temporal_manifest.loc[
    temporal_manifest["split"].eq("test")
]

assert len(test_rows) == 1
test_row = test_rows.iloc[0]

# Notebook 05 consumes one frozen test interval. Reject unsupported layouts explicitly.
def interval_for(snapshot):
    label_row = label_manifest.loc[label_manifest["feature_snapshot"].eq(snapshot)].iloc[0]
    return f"{snapshot}_to_{label_row.outcome_snapshot}"
validation_snapshots = label_manifest.loc[label_manifest["split"].eq("validation"), "feature_snapshot"]
assert len(validation_snapshots) == 1, "Evaluation plan requires one validation interval"


evaluation_plan = {
    "primary_model": primary_key,
    "comparator_model": comparator_key,
    "selection_basis": (
        "Exploratory validation: primary selected by mean average "
        "precision across benign and pathogenic reclassification."
    ),
    "training_intervals": [interval_for(value) for value in label_manifest.loc[label_manifest["split"].eq("train"), "feature_snapshot"]],
    "validation_interval": interval_for(validation_snapshots.iloc[0]),
    "test_interval": interval_for(test_row["feature_snapshot"]),
    "refitted_on_validation": False,
    "selection_scores": selection_table.to_dict(orient="records"),
    "validation_variants_seen_in_training": overlap_count,
    "test_history": "This interval was evaluated in earlier experiments; it is not a fresh untouched external test.",
    "test_files": {
        "baseline_features": test_row["feature_file"],
        "temporal_features": test_row["temporal_file"],
        "labels": test_row["label_file"]
    },
    "expected_test_rows": int(test_row["rows"]),
    "feature_config": temporal_config,
    "primary_metric": "Mean average precision across the two reclassification targets",
    "additional_metrics": [
        "Per-target average precision and prevalence",
        "Macro F1 and balanced accuracy",
        "Per-class precision and recall",
        "Confusion matrix",
        "Precision and recall at fixed review sizes"
    ],
    "review_sizes": [100, 1000, 5000],
    "tie_break_seed": 42,
    "test_policy": (
        "Evaluate frozen pipelines without tuning on test results. "
        "If later work uses these results to change the model, "
        "this interval is no longer an untouched final test."
    ),
    "scores_are_calibrated_probabilities": False,
    "scope": (
        "Eligible next-snapshot outcomes among starting VUS variants; "
        "not clinical pathogenicity diagnosis."
    ),
    "sklearn_version": sklearn.__version__,
    "deployment_status": "Research experiment; not approved for deployment"
}

plan_file = evaluation_bundle / "evaluation_plan.json"
plan_file.write_text(
    json.dumps(evaluation_plan, indent=2),
    encoding="utf-8"
)

assert json.loads(
    plan_file.read_text(encoding="utf-8")
) == evaluation_plan

print("MODELS FROZEN — SAVE AND RELOAD CHECKS PASSED")
print("\nEvaluation bundle:")
print(evaluation_bundle)
print("\nNo test predictions were made in this cell.")
import hashlib
provenance = []
for row in temporal_manifest.loc[temporal_manifest["split"].isin(["train", "validation"])].itertuples(index=False):
    for source in [feature_folder / row.feature_file, temporal_folder / row.temporal_file, label_folder / row.label_file]:
        digest = hashlib.sha256()
        with source.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        provenance.append({"file": str(source.relative_to(PROJECT_ROOT)), "sha256": digest.hexdigest()})
(evaluation_bundle / "development_inputs.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")


                        model  mean_average_precision
0  temporal_gradient_boosting                0.009129
1           temporal_balanced                0.007122
2         temporal_unweighted                0.007057
MODELS FROZEN — SAVE AND RELOAD CHECKS PASSED

Evaluation bundle:
D:\GeneVISTA\models\temporal_v1\20260923_100012_125957

No test predictions were made in this cell.
